# Structure of the raw SFTR securities lending data

Source table `crp_sftds_ecb.trade_states_securitieslending`. Each section asks one question about the raw data, runs the query that answers it and follows up on what the answer leaves open.

In [45]:
import pyodbc
import pandas as pd
import numpy as np

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

## 2. What is one row?

Follow one trade through the table. `tec_ruti` identifies a trade across both reporting sides.

In [46]:
query = f"""

SELECT reference_period, reporting_cpty_id, other_cpty_id, counterparty_side, uti
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti = '549300GWM9UJLZCGSN04R0MUWSFPU8MPRO8K5P83F2KUVLIFOYYYLY7A86QTJNMN76CPND8WN28Z'
ORDER BY reference_period

"""
df = pd.read_sql_query(query, cnxn)
df['reference_period'].value_counts()

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\108234014.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


reference_period
2026-06-04    2
2026-07-28    2
2026-07-14    2
2026-07-15    2
2026-07-16    2
2026-07-17    2
2026-07-20    2
2026-07-21    2
2026-07-22    2
2026-07-23    2
2026-07-24    2
2026-07-27    2
2026-07-29    2
2026-07-10    2
2026-07-30    2
2026-07-31    2
2026-08-03    2
2026-08-04    2
2026-08-05    2
2026-08-06    2
2026-08-07    2
2026-08-10    2
2026-08-11    2
2026-08-12    2
2026-07-13    2
2026-07-09    2
2026-06-05    2
2026-06-22    2
2026-06-08    2
2026-06-09    2
2026-06-10    2
2026-06-11    2
2026-06-12    2
2026-06-15    2
2026-06-16    2
2026-06-17    2
2026-06-18    2
2026-06-19    2
2026-06-23    2
2026-07-08    2
2026-06-24    2
2026-06-25    2
2026-06-26    2
2026-06-29    2
2026-06-30    2
2026-07-01    2
2026-07-02    2
2026-07-03    2
2026-07-06    2
2026-07-07    2
2026-08-13    2
Name: count, dtype: int64

Answer. One row per reporting side and day, so a trade reported by both sides gives two rows per day. The two legs on one day share the UTI, with reporting and other counterparty swapped and opposite `counterparty_side`.

In [47]:
df.head(2)

,reference_period,reporting_cpty_id,other_cpty_id,counterparty_side,uti
0,2026-06-04,R0MUWSFPU8MPRO8K5P83,549300GWM9UJLZCGSN04,GIVE,F2KUVLIFOYYYLY7A86QTJNMN76CPND8WN28Z
1,2026-06-04,549300GWM9UJLZCGSN04,R0MUWSFPU8MPRO8K5P83,TAKE,F2KUVLIFOYYYLY7A86QTJNMN76CPND8WN28Z


## 3. How are rows grouped by trade and day?

How many rows share a `tec_ruti` on one day, how many of them carry the best value leg flag, and how many distinct event dates do they have?

In [48]:
query = f"""

SELECT n_rows, n_best, n_dates, COUNT(*) AS n_groups
FROM (
  SELECT tec_ruti, reference_period,
         COUNT(*) AS n_rows,
         SUM(CASE WHEN tec_best_value_leg = 1 THEN 1 ELSE 0 END) AS n_best,
         COUNT(DISTINCT event_date) AS n_dates
  FROM crp_sftds_ecb.trade_states_securitieslending
  GROUP BY tec_ruti, reference_period
) g
GROUP BY n_rows, n_best, n_dates
ORDER BY n_groups DESC

"""
df = pd.read_sql_query(query, cnxn)
df.head(30)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\1495388006.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,n_rows,n_best,n_dates,n_groups
0,1,0,1,121625857
1,2,1,1,16065709
2,2,1,2,4972640
3,1,1,1,1868
4,3,0,2,78
5,139474,0,960,1
6,134656,0,955,1
7,126301,0,950,1
8,127244,0,947,1
9,138687,0,965,1


Answer. Single legs (one row, no flag), pairs (two rows, exactly one flag) and a handful of triples. The best value leg flag is only set within pairs, so it cannot be used as a filter on its own.

In [49]:
df[df['n_rows'] <= 3]

,n_rows,n_best,n_dates,n_groups
0,1,0,1,121625857
1,2,1,1,16065709
2,2,1,2,4972640
3,1,1,1,1868
4,3,0,2,78


What are the lines with more than 100k rows per day? A NULL `tec_ruti` forms one group per day in a `GROUP BY`, so these are the rows without a key. What are they?

In [50]:
query = f"""

SELECT reference_period, action_type,
       CASE WHEN uti IS NULL THEN 1 ELSE 0 END AS uti_missing,
       CASE WHEN loan_security_id IS NULL THEN 1 ELSE 0 END AS isin_missing,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NULL OR tec_ruti = ''
GROUP BY 1, 2, 3, 4
ORDER BY reference_period, n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\3913030310.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,action_type,uti_missing,isin_missing,n
0,2026-06-01,COLU,1,1,129554
1,2026-06-02,COLU,1,1,130317
2,2026-06-03,COLU,1,1,130625
3,2026-06-04,COLU,1,1,132193
4,2026-06-05,COLU,1,1,127244
...,...,...,...,...,...
73,2026-09-10,COLU,1,1,134582
74,2026-09-11,COLU,1,1,133509
75,2026-09-14,COLU,1,1,130676
76,2026-09-15,COLU,1,1,130152


In [51]:
df['action_type'].unique()

array(['COLU'], dtype=object)

In [52]:
df['uti_missing'].unique()

array([1], dtype=int64)

In [53]:
df['isin_missing'].unique()

array([1], dtype=int64)

Answer. Collateral updates on a net exposure basis. They carry no UTI and no ISIN, so they belong to a counterparty pair rather than to a loan, and their event dates are the dates of the last collateral report per pair, which can lie years back. The loan table drops them with `tec_ruti IS NOT NULL`.

Is `tec_surrogate_key` unique per row? It is the row key of the table.

In [54]:
query = f"""

SELECT COUNT(*), COUNT(DISTINCT tec_surrogate_key)
FROM crp_sftds_ecb.trade_states_securitieslending

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\247840212.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,expr_1,expr_2
0,174315664,174315664


Answer. Yes.

## 4. Where is the collateral?

For which loans do the arrays hold anything? Cross the two flags with whether the arrays are filled.

In [55]:
query = f"""

SELECT collateralisation_net_exposure, uncollateralised_flag,
       CASE WHEN number_collateral_securities > 0 THEN 1 ELSE 0 END AS has_sec,
       CASE WHEN number_collateral_cash > 0 THEN 1 ELSE 0 END AS has_cash,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NOT NULL AND tec_ruti <> ''
GROUP BY 1, 2, 3, 4
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\2575286503.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,collateralisation_net_exposure,uncollateralised_flag,has_sec,has_cash,n
0,True,False,0,0,115704011
1,False,False,0,1,31033129
2,None,True,0,0,10672004
3,False,False,0,0,2216146
4,False,False,1,0,1813612
5,True,False,1,0,1194875
6,True,False,0,1,1038146
7,False,False,1,1,19174
8,None,False,0,0,9917
9,True,False,1,1,3643


Answer. The flag and the arrays describe two layers of collateral. `collateralisation_net_exposure` says that margining runs on the net exposure of the counterparty pair, the arrays say whether this loan also has collateral allocated to it, and SFTR allows both at once (ESMA guidelines, section 5.4.7). So the combinations read as follows.

* Net exposure with empty arrays is the pool only case and the most frequent one. Typical for agency lending, where one collateral pool per borrower and lender pair covers many small loans. The pool sits in the UTI less rows of section 3, the loan row cannot show it.
* Net exposure with cash or securities is a loan with its own collateral that is additionally margined on a net basis, the case in ESMA example 5.4.7.1.
* Both flags false with cash or securities is plain trade level collateral. Cash and securities together is mixed collateral, rare but legitimate.
* Both flags false with empty arrays is basket collateral or missing collateral, see the next query.
* Uncollateralised loans carry nothing by definition. Rows with both flags NULL are incomplete reports.

What about loans with both flags false but neither securities nor cash?

In [56]:
query = f"""

SELECT CASE WHEN collateral_basket_id IS NULL THEN 0 ELSE 1 END AS has_basket, COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NOT NULL AND tec_ruti <> ''
  AND collateralisation_net_exposure = FALSE AND uncollateralised_flag = FALSE
  AND COALESCE(number_collateral_securities, 0) = 0
  AND COALESCE(number_collateral_cash, 0) = 0
GROUP BY 1

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\2663623859.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,has_basket,n
0,1,2087106
1,0,129040


Answer. Nearly all of them reference a collateral basket, `collateral_basket_id`. The rest have missing collateral.

How many pieces of securities collateral does a loan carry?

In [57]:
query = f"""

SELECT number_collateral_securities AS n_sec, COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE number_collateral_securities > 0
GROUP BY 1
ORDER BY 1
LIMIT 30

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\1949470445.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,n_sec,n
0,1,3952568
1,2,2528437
2,3,655308
3,4,486641
4,5,397810
5,6,351930
6,7,222543
7,8,166488
8,9,123015
9,10,109055


What does one array element look like? The comma between the table and `t.collateral_security` unnests the array, one row per element.

In [58]:
query = f"""

SELECT t.reference_period, t.uti, e.id, e.market_value_eur, e.haircut_margin, e.security_type, e.quality
FROM crp_sftds_ecb.trade_states_securitieslending t, t.collateral_security e
WHERE uti IS NOT NULL
LIMIT 10

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\1296614772.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,uti,id,market_value_eur,haircut_margin,security_type,quality
0,2026-08-25,20260821PAT393398298,ES0000012I32,8.795685e+07,0.0,GOVS,INVG
1,2026-08-25,20260821PAT393398298,IT0005436693,8.084867e+08,0.0,GOVS,INVG
2,2026-08-25,E02ZZZG04004ZZZ1044502055,GB0006731235,3.455252e+06,0.0,MEQU,NOAP
3,2026-08-25,F2KUVLIFOYBF232W4YLLT23PMM4ALA6XU8GP,GB00BNC5T391,2.400991e+05,0.0,MEQU,NOAP
4,2026-08-25,FR969500CJCTMI93QJKK89E3055420,XS1917358621,1.092600e+02,4.8,FIDE,INVG
5,2026-08-25,F2KUVLIFOYF3F6BCUD2DE2E3F4JTNP38LVCP,FR0011726835,1.961423e+07,0.0,MEQU,NOAP
6,2026-08-25,20260821PAT393416318,ES0000012I32,8.795685e+07,0.0,GOVS,INVG
7,2026-08-25,20260821PAT393416318,IT0005436693,8.084867e+08,0.0,GOVS,INVG
8,2026-08-25,20260821PAT393431901,IT0005436693,8.084867e+08,0.0,GOVS,INVG
9,2026-08-25,20260821PAT393431901,ES0000012I32,8.795685e+07,0.0,GOVS,INVG


In [59]:
query = f"""

SELECT t.reference_period, t.uti, e.amount, e.amount_currency, e.amount_eur, e.haircut_margin
FROM crp_sftds_ecb.trade_states_securitieslending t, t.collateral_cash e
LIMIT 10

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\2976493077.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,reference_period,uti,amount,amount_currency,amount_eur,haircut_margin
0,2026-08-05,5493001B9LDFCQUY927320260804XU13242830ASL12503...,-6303.15,EUR,-6.303150e+03,0.000000
1,2026-08-05,20260722LV62021858000952,7489.60,EUR,7.489600e+03,20.000000
2,2026-08-05,BNPBOLIVAR3254129120260723231136,-2586591.37,EUR,-2.586591e+06,0.000000
3,2026-08-05,5493001B9LDFCQUY927320260804U9497780SL125059150,-10296.00,USD,-8.913514e+03,0.000000
4,2026-08-05,BNPBOLIVARY85390120260305071435,29972.36,EUR,2.997236e+04,0.000000
5,2026-08-05,815600E4E6DCD2D25E30A20260804IT000525320500023...,-508.25,EUR,-5.082500e+02,0.000000
6,2026-08-05,5493001B9LDFCQUY927320260804U10134753SL125051784,-10744.00,USD,-9.301359e+03,0.000000
7,2026-08-05,5493001B9LDFCQUY927320260804U7455236SL125051824,-4590.00,USD,-3.973682e+03,0.000000
8,2026-08-05,5493001B9LDFCQUY927320260804U16522917SL125050160,-17100.00,USD,-1.480391e+04,0.000000
9,2026-08-05,BNP549300KUN9K9K32C6D97XFALCON20260601288314L,-636020.00,USD,-5.506190e+05,1.960784


Why are market values in the array elements sometimes negative? Cross the sign with the reporting side, separately for loan rows and for the UTI less pool rows.

In [60]:
query = f"""

SELECT CASE WHEN t.tec_ruti IS NULL OR t.tec_ruti = '' THEN 'pool' ELSE 'loan' END AS row_type,
       t.counterparty_side,
       CASE WHEN e.market_value_eur < 0 THEN 'negative'
            WHEN e.market_value_eur > 0 THEN 'positive' ELSE 'zero_or_null' END AS sign,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending t, t.collateral_security e
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\428617703.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,row_type,counterparty_side,sign,n
0,loan,GIVE,negative,1176021
1,loan,GIVE,positive,4898612
2,loan,GIVE,zero_or_null,15175
3,loan,TAKE,negative,131964
4,loan,TAKE,positive,884579
5,loan,TAKE,zero_or_null,19638
6,pool,None,negative,112926439
7,pool,None,positive,133489682
8,pool,None,zero_or_null,1072493


In [61]:
query = f"""

SELECT CASE WHEN t.tec_ruti IS NULL OR t.tec_ruti = '' THEN 'pool' ELSE 'loan' END AS row_type,
       t.counterparty_side,
       CASE WHEN e.amount_eur < 0 THEN 'negative'
            WHEN e.amount_eur > 0 THEN 'positive' ELSE 'zero_or_null' END AS sign,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending t, t.collateral_cash e
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\2885550702.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,row_type,counterparty_side,sign,n
0,loan,GIVE,negative,15607825
1,loan,GIVE,positive,7435969
2,loan,GIVE,zero_or_null,254477
3,loan,TAKE,negative,3725016
4,loan,TAKE,positive,4999289
5,loan,TAKE,zero_or_null,86668
6,pool,None,negative,200024
7,pool,None,positive,379182
8,pool,None,zero_or_null,97303


Answer. There is no clean convention on the loan rows. Negative values appear on both sides, for cash on 68 percent of GIVE elements and 43 percent of TAKE elements, for securities on 19 and 13 percent. Some reporters apply the perspective sign to trade level collateral, others do not, so the sign carries no reliable direction. The pool rows carry no `counterparty_side`, as ESMA prescribes for net exposure collateral, so there the sign is the only indicator of direction, negative for the provider and positive for the taker.

The element example showed the same ISINs with identical values on several UTIs of one day. Are the arrays on net exposure loans the collateral pool repeated on each loan? Count, per day and counterparty pair, how many loans share an identical element.

In [62]:
query = f"""

SELECT collateralisation_net_exposure, n_loans_sharing, COUNT(*) AS n_elements
FROM (
  SELECT t.collateralisation_net_exposure, t.reference_period,
         t.reporting_cpty_id, t.other_cpty_id, e.id, e.market_value_eur,
         COUNT(DISTINCT t.uti) AS n_loans_sharing
  FROM crp_sftds_ecb.trade_states_securitieslending t, t.collateral_security e
  WHERE t.tec_ruti IS NOT NULL AND t.tec_ruti <> ''
  GROUP BY 1, 2, 3, 4, 5, 6
) s
GROUP BY 1, 2
ORDER BY 1, 2;

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\4083246349.py:16: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,collateralisation_net_exposure,n_loans_sharing,n_elements
0,False,1,2344025
1,False,2,413106
2,False,3,102528
3,False,4,33640
4,False,5,12481
5,False,6,5443
6,False,7,2778
7,False,8,1492
8,False,9,774
9,False,10,503


In [63]:
df['n_rows'] = df['n_elements'] * df['n_loans_sharing']
df['shared'] = df['n_loans_sharing'] > 1
df.groupby(['collateralisation_net_exposure', 'shared'])['n_rows'].sum().unstack()

shared,False,True
collateralisation_net_exposure,,
False,2344025,1418406
True,1917371,1387860


Answer. Only partly. About 40 percent of the element rows on net exposure loans are shared by several loans of the same pair and day, the rest belong to one loan. Trade level loans show almost the same share, which equal sized allocations of one borrow across beneficial owners can explain. So the arrays do not give a reliable per loan collateral value, and the cleaning query keeps only the collateral type, not the values.

## 5. How is the price of the loan stored?

Which rate columns are filled for fixed rebates, floating rebates and fee based loans?

In [64]:
query = f"""

SELECT rebate_rate_type, COUNT(*) AS n,
       COUNT(fxd_rebate_rate) AS n_fixed_rate,
       COUNT(flt_rebate_rate) AS n_float_index,
       COUNT(rebate_rate_derived_sdw) AS n_derived_rate,
       COUNT(lending_fee) AS n_fee
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NOT NULL AND tec_ruti <> ''
GROUP BY 1

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\1725497723.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,rebate_rate_type,n,n_fixed_rate,n_float_index,n_derived_rate,n_fee
0,None,131360125,0,0,0,128320407
1,Floating,2739403,0,1180844,962822,234369
2,Fixed,29605129,29605129,0,29605129,2735680


Is `rebate_rate_derived_sdw` the floating index plus the spread?

In [65]:
query = f"""

SELECT rebate_rate_type, COUNT(*) AS n,
       APPX_MEDIAN(rebate_rate_derived_sdw
                   - (flt_rebate_rate_value_sdw + flt_rebate_rate_spread_basispoints / 100)) AS med_diff,
       MIN(rebate_rate_derived_sdw) AS min_rate, MAX(rebate_rate_derived_sdw) AS max_rate
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE flt_rebate_rate IS NOT NULL
GROUP BY 1

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\614880981.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,rebate_rate_type,n,med_diff,min_rate,max_rate
0,Floating,1180844,0.0,3.3,11.56


Answer. Yes, the median difference is zero, and rates are in percent per annum.

## 6. What else does the cleaning query filter on?

Does the snapshot contain position level rows or terminated trades? `level` is TCTN for a trade and PSTN for a CCP position. When cleared trades are netted into a position, the components are reported with action type POSC and stop being outstanding, so position and components must not be counted together.

In [66]:
query = f"""

SELECT level, action_type, cleared, COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NOT NULL AND tec_ruti <> ''
GROUP BY 1, 2, 3
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\158662586.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,level,action_type,cleared,n
0,TCTN,VALU,False,74268955
1,TCTN,MODI,False,62475397
2,TCTN,NEWT,False,21438762
3,TCTN,COLU,False,3005524
4,TCTN,VALU,True,1464061
5,TCTN,CORR,False,587687
6,PSTN,MODI,True,236058
7,TCTN,NEWT,True,117382
8,TCTN,MODI,True,76860
9,PSTN,VALU,True,30142


Answer. The snapshot holds only outstanding trades, there are no ETRM and no POSC rows. Positions are rare, about 270 thousand rows or 0.2 percent, nearly all cleared, and the last action on a loan row is mostly a valuation update or a modification. Since no components are flagged as POSC there is nothing to remove, positions and trades can be told apart through `level`.

Which rows have no ISIN for the security on loan? The cleaning query requires one, which is meant to remove commodity loans only.

In [67]:
query = f"""

SELECT loan_asset_type,
       CASE WHEN loan_security_id IS NULL THEN 1 ELSE 0 END AS isin_missing,
       COUNT(*) AS n
FROM crp_sftds_ecb.trade_states_securitieslending
WHERE tec_ruti IS NOT NULL AND tec_ruti <> ''
GROUP BY 1, 2
ORDER BY n DESC

"""
df = pd.read_sql_query(query, cnxn)
df

C:\Users\hermesf\AppData\Local\Temp\ipykernel_20072\1520383755.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, cnxn)


,loan_asset_type,isin_missing,n
0,Security,0,163693734
1,Commodity,1,10923


Answer. Every securities loan has an ISIN. The rows without one are the commodity loans, about eleven thousand, so the condition removes exactly those.

The cleaning queries built from these checks are in `sec_lending_clean_query.txt` (loans) and `sec_lending_collateral_query.txt` (collateral pieces).